In [ ]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_log_error as RMSE


data_path = "train.csv"
data = pd.read_csv(data_path)
target_thing = np.log1p(data["hours_spent"])

input_data = data.drop(columns=[
    "project_id",
    "hours_spent",
    "log_hours",
    "hours_bucket"
])


qualitative_features = input_data.select_dtypes(include=["object", "string", "category"]).columns.tolist() 
#print(qualitative_features) => ['commit_cadence', 'primary_language', 'framework', 'has_custom_art', 'hosting_type', 'is_mobile_responsive', 'project_category', 'is_multiplayer', 'mentions_ai_api', 'is_clone_or_tutorial', 'author_prior_projects', 'author_account_age_years']

for col in qualitative_features: #fill in missing stuff just in case
    input_data[col] = input_data[col].fillna("Unknown").astype(str)

input_training, input_validation, output_training, output_validation = train_test_split(input_data, target_thing, test_size=0.2, random_state=42)

magicModel = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.01,
    depth=6,
    loss_function="RMSE",
    verbose=100,
    early_stopping_rounds=100
)

magicModel.fit(
    input_training,
    output_training,
    cat_features=qualitative_features,
    eval_set=(input_validation, output_validation)
)

predictions=np.expm1(magicModel.predict(input_validation))
predictions = np.maximum(predictions, 0) #no negatives

true_hours = np.expm1(output_validation)

score=RMSE(true_hours,predictions)

print("the score:", score)

#diagnostics

how_important = pd.DataFrame({
    "feature": input_data.columns,
    "importance": magicModel.feature_importances_
})

print("correlations")
print(how_important.head(15).to_string(index=False))

In [1]:
import pandas as pd
import numpy as np

data_path = "train.csv"
data = pd.read_csv(data_path)
target_thing = np.log1p(data["hours_spent"])

print("Projects per categoy")
print(data["project_category"].value_counts())

print("average horus per category")
print(data.groupby("project_category")["hours_spent"]
      .agg(["count", "mean", "median","std", "min", "max"])
      .sort_values("count", ascending=False))


Projects per categoy
project_category
game       1687
app         961
website     939
tool        697
bot         128
other        83
Name: count, dtype: int64
average horus per category
                  count       mean  median        std  min    max
project_category                                                 
game               1687  11.486011     6.0  18.495224  0.1  140.7
app                 961  14.663684     8.0  19.814738  0.2  140.7
website             939   7.617359     3.1  13.445596  0.1  140.7
tool                697  16.230560     8.8  24.041063  0.1  140.7
bot                 128  12.771094     7.3  19.631339  0.2  140.7
other                83  12.039759     6.0  15.518376  0.7   85.0
